# 02 — Master Data Creation

Merge **Customer Details**, **Loan Details**, and **Payment Details** into a single master dataset at **payment-level granularity**.

### Merge Strategy
1. **Prepare Customer Data** — Separate primary vs co-applicants, aggregate co-applicant features per loan
2. **Merge Customer + Loan** — LEFT JOIN loan details (base) with prepared customer data
3. **Filter** — Remove loans that have no payment records (cannot contribute to credit risk analysis)
4. **Merge with Payment** — INNER JOIN payment details with loan+customer table (only fully matched records)

### Outputs
- `master_data/master_payment_level.csv`
- `master_data/master_data_summary.xlsx`

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) != 'Five_star_customer_data' else os.getcwd()
# Override if running from the project root
if not os.path.exists(os.path.join(BASE_DIR, 'customer_loan_data')):
    BASE_DIR = os.getcwd()

OUTPUT_DIR = os.path.join(BASE_DIR, 'master_data')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Base directory: {BASE_DIR}')
print(f'Output directory: {OUTPUT_DIR}')

Base directory: /Users/shiva/Downloads/Five_star_customer_data
Output directory: /Users/shiva/Downloads/Five_star_customer_data/master_data


## 1. Load Source Data

In [2]:
# Load all 3 source files
customer_df = pd.read_csv(os.path.join(BASE_DIR, 'customer_loan_data', 'five_star_customer_details_masked.csv'))
loan_df = pd.read_csv(os.path.join(BASE_DIR, 'customer_loan_data', 'FS_loan_details_masked.csv'))
payment_df = pd.read_csv(os.path.join(BASE_DIR, 'pymnt_data', 'five_star_pymt_details_masked.csv'))

print('=== Source Data Shapes ===')
print(f'Customer Details : {customer_df.shape}  | Unique loans: {customer_df["masked_loan_id"].nunique():,}')
print(f'Loan Details     : {loan_df.shape}  | Unique loans: {loan_df["masked_loan_id"].nunique():,}')
print(f'Payment Details  : {payment_df.shape} | Unique loans: {payment_df["masked_loan_id"].nunique():,}')

=== Source Data Shapes ===
Customer Details : (1969639, 13)  | Unique loans: 630,743
Loan Details     : (630632, 21)  | Unique loans: 630,632


Payment Details  : (10218442, 22) | Unique loans: 601,328


In [3]:
print('\n=== Customer Columns ===')
print(customer_df.columns.tolist())
print('\n=== Loan Columns ===')
print(loan_df.columns.tolist())
print('\n=== Payment Columns ===')
print(payment_df.columns.tolist())


=== Customer Columns ===
['Unnamed: 0', 'fileno', 'customer_age', 'gender', 'employment_status', 'customer_income', 'income_to_emi_ratio', 'customer_segment', 'prior_loans_count', 'prior_delinquency_history', 'customer_geography_type', 'caopp_data_flag', 'masked_loan_id']

=== Loan Columns ===
['Unnamed: 0', 'fileno', 'loan_origination_date', 'scheduled_maturity_date', 'tenure_total', 'product_type', 'secured_flag', 'interest_rate', 'emi_frequency', 'disbursed_amount', 'principal_outstanding', 'ltv', 'bureau_score_origination', 'restructured_flag', 'moratorium_flag', 'branch', 'state', 'repayment_start_date', 'Sourcing channel', 'scheme', 'masked_loan_id']

=== Payment Columns ===
['Unnamed: 0', 'fileno', 'status', 'closure date', 'maturity date', 'installment number', 'installment_due_date', 'emi_amount', 'payment_date', 'paid EMI', 'unpaid EMI', 'payment_mode', 'bounce_flag', 'bounced date', 'number_of_paid_emis', 'number_of_unpaid_emis', 'dpd (max)', 'bucket_prior', 'bucket_current

## 2. Step 1 — Prepare Customer Data (Aggregate Co-Applicants)

In [4]:
# Check caopp_data_flag distribution
print('caopp_data_flag value counts:')
print(customer_df['caopp_data_flag'].value_counts())
print(f'\nTotal unique loans: {customer_df["masked_loan_id"].nunique():,}')

caopp_data_flag value counts:
caopp_data_flag
Yes    1967561
No        2078
Name: count, dtype: int64

Total unique loans: 630,743


In [5]:
# Separate primary applicants (caopp_data_flag='No') and co-applicants (caopp_data_flag='Yes')
primary_df = customer_df[customer_df['caopp_data_flag'] == 'No'].copy()
coapplicant_df = customer_df[customer_df['caopp_data_flag'] == 'Yes'].copy()

print(f'Primary applicant rows   : {len(primary_df):,} | Unique loans: {primary_df["masked_loan_id"].nunique():,}')
print(f'Co-applicant rows        : {len(coapplicant_df):,} | Unique loans: {coapplicant_df["masked_loan_id"].nunique():,}')

Primary applicant rows   : 2,078 | Unique loans: 2,078
Co-applicant rows        : 1,967,561 | Unique loans: 628,665


In [6]:
# Find loans that have ONLY co-applicant rows (no primary applicant row)
loans_with_primary = set(primary_df['masked_loan_id'].unique())
loans_with_coapp_only = set(coapplicant_df['masked_loan_id'].unique()) - loans_with_primary
all_customer_loans = set(customer_df['masked_loan_id'].unique())

print(f'Loans with primary applicant row : {len(loans_with_primary):,}')
print(f'Loans with ONLY co-applicant rows: {len(loans_with_coapp_only):,}')
print(f'Total unique loans in customer   : {len(all_customer_loans):,}')

Loans with primary applicant row : 2,078
Loans with ONLY co-applicant rows: 628,665
Total unique loans in customer   : 630,743


In [7]:
# For loans with only co-applicant rows, pick the first row as the primary applicant proxy
coapp_only_rows = coapplicant_df[coapplicant_df['masked_loan_id'].isin(loans_with_coapp_only)]
proxy_primary = coapp_only_rows.groupby('masked_loan_id').first().reset_index()
proxy_primary['caopp_data_flag'] = 'No'  # mark as primary proxy

print(f'Proxy primary applicant rows created: {len(proxy_primary):,}')

# Combine real primary + proxy primary
primary_combined = pd.concat([primary_df, proxy_primary], ignore_index=True)

# Remove proxy rows from co-applicant pool to avoid double-counting
# For loans that only had co-applicants, we used the first row as primary.
# The remaining co-applicant rows (if any) for those loans should still be aggregated.
# We need to remove only the specific rows used as proxies.
proxy_ids = set(proxy_primary['masked_loan_id'])
coapp_remaining_from_proxy_loans = coapp_only_rows.groupby('masked_loan_id').apply(
    lambda x: x.iloc[1:] if len(x) > 1 else x.iloc[0:0]
).reset_index(drop=True)

# Co-applicants = original co-applicants for loans WITH a primary + remaining for proxy loans
coapp_for_loans_with_primary = coapplicant_df[coapplicant_df['masked_loan_id'].isin(loans_with_primary)]
coapplicant_final = pd.concat([coapp_for_loans_with_primary, coapp_remaining_from_proxy_loans], ignore_index=True)

print(f'\nFinal primary rows      : {len(primary_combined):,}')
print(f'Final co-applicant rows : {len(coapplicant_final):,}')

# Deduplicate primary: keep first per loan
primary_combined = primary_combined.drop_duplicates(subset='masked_loan_id', keep='first')
print(f'Primary after dedup     : {len(primary_combined):,}')

Proxy primary applicant rows created: 628,665



Final primary rows      : 630,743
Final co-applicant rows : 1,338,896
Primary after dedup     : 630,743


In [8]:
# Aggregate co-applicant features per loan
def aggregate_co_applicants(df):
    """Aggregate co-applicant data per masked_loan_id."""
    if len(df) == 0:
        return pd.DataFrame()
    
    agg_dict = {
        'masked_loan_id': 'first',
    }
    
    result = df.groupby('masked_loan_id').agg(
        num_co_applicants=('masked_loan_id', 'size'),
        max_co_applicant_age=('customer_age', 'max'),
        min_co_applicant_age=('customer_age', 'min'),
        mean_co_applicant_age=('customer_age', 'mean'),
        co_applicant_income_sum=('customer_income', 'sum'),
        co_applicant_income_max=('customer_income', 'max'),
    ).reset_index()
    
    # Mode of gender among co-applicants
    gender_mode = df.groupby('masked_loan_id')['gender'].agg(
        lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan
    ).reset_index()
    gender_mode.columns = ['masked_loan_id', 'co_applicant_gender_mode']
    
    result = result.merge(gender_mode, on='masked_loan_id', how='left')
    return result

coapp_agg = aggregate_co_applicants(coapplicant_final)
print(f'Co-applicant aggregated rows: {len(coapp_agg):,}')
if len(coapp_agg) > 0:
    print('\nAggregated columns:', coapp_agg.columns.tolist())
    display(coapp_agg.head())

Co-applicant aggregated rows: 0


In [9]:
# Merge primary applicant with co-applicant aggregates
# Drop the unnamed index column and caopp_data_flag from primary
cols_to_drop_primary = [c for c in primary_combined.columns if c.startswith('Unnamed')]
if 'caopp_data_flag' in primary_combined.columns:
    cols_to_drop_primary.append('caopp_data_flag')
primary_clean = primary_combined.drop(columns=cols_to_drop_primary, errors='ignore')

# Rename primary applicant columns for clarity
rename_map = {
    'customer_age': 'primary_customer_age',
    'gender': 'primary_gender',
    'employment_status': 'primary_employment_status',
    'customer_income': 'primary_customer_income',
    'income_to_emi_ratio': 'primary_income_to_emi_ratio',
}
primary_clean = primary_clean.rename(columns=rename_map)

# Merge with co-applicant aggregates
if len(coapp_agg) > 0:
    customer_prepared = primary_clean.merge(coapp_agg, on='masked_loan_id', how='left')
    # Fill num_co_applicants with 0 for loans without co-applicants
    customer_prepared['num_co_applicants'] = customer_prepared['num_co_applicants'].fillna(0).astype(int)
else:
    customer_prepared = primary_clean.copy()
    customer_prepared['num_co_applicants'] = 0

print(f'Customer prepared shape: {customer_prepared.shape}')
print(f'Unique loans: {customer_prepared["masked_loan_id"].nunique():,}')
print(f'\nColumns: {customer_prepared.columns.tolist()}')
display(customer_prepared.head())

Customer prepared shape: (630743, 12)


Unique loans: 630,743

Columns: ['fileno', 'primary_customer_age', 'primary_gender', 'primary_employment_status', 'primary_customer_income', 'primary_income_to_emi_ratio', 'customer_segment', 'prior_loans_count', 'prior_delinquency_history', 'customer_geography_type', 'masked_loan_id', 'num_co_applicants']


,fileno,primary_customer_age,primary_gender,primary_employment_status,primary_customer_income,primary_income_to_emi_ratio,customer_segment,prior_loans_count,prior_delinquency_history,customer_geography_type,masked_loan_id,num_co_applicants
0,5028651,NaN,NaN,NaN,NaN,NaN,Low,1.0,90+,NaN,LN_tUsRhqumEls=,0
1,5028652,NaN,NaN,NaN,NaN,NaN,Low,1.0,90+,NaN,LN_WTJa7Vs8T4k=,0
2,5028653,NaN,NaN,NaN,NaN,NaN,Low,1.0,0,NaN,LN_adeDnASX970=,0
3,5028654,NaN,NaN,NaN,NaN,NaN,Low,1.0,0,NaN,LN_gQP8PXbkW4E=,0
4,5028655,NaN,NaN,NaN,NaN,NaN,Low,1.0,90+,NaN,LN_zqWLp34s8IU=,0


## 3. Step 2 — Merge Customer + Loan Details

In [10]:
# Clean loan_df: drop unnamed index column
loan_clean = loan_df.drop(columns=[c for c in loan_df.columns if c.startswith('Unnamed')], errors='ignore')

print(f'Loan details shape : {loan_clean.shape}')
print(f'Columns: {loan_clean.columns.tolist()}')

Loan details shape : (630632, 20)
Columns: ['fileno', 'loan_origination_date', 'scheduled_maturity_date', 'tenure_total', 'product_type', 'secured_flag', 'interest_rate', 'emi_frequency', 'disbursed_amount', 'principal_outstanding', 'ltv', 'bureau_score_origination', 'restructured_flag', 'moratorium_flag', 'branch', 'state', 'repayment_start_date', 'Sourcing channel', 'scheme', 'masked_loan_id']


In [11]:
# LEFT JOIN: Loan (base) + Customer (prepared)
# Identify overlapping columns (besides the join key)
common_cols = set(loan_clean.columns) & set(customer_prepared.columns) - {'masked_loan_id'}
print(f'Overlapping columns (will use suffixes): {common_cols}')

loan_customer = loan_clean.merge(
    customer_prepared,
    on='masked_loan_id',
    how='left',
    suffixes=('_loan', '_cust')
)

# For overlapping columns like 'fileno', keep the loan version and drop customer version
cols_to_drop = [c + '_cust' for c in common_cols if c + '_cust' in loan_customer.columns]
# Rename loan versions back to original names
cols_to_rename = {c + '_loan': c for c in common_cols if c + '_loan' in loan_customer.columns}

loan_customer = loan_customer.drop(columns=cols_to_drop, errors='ignore')
loan_customer = loan_customer.rename(columns=cols_to_rename)

print(f'\nLoan + Customer merged shape: {loan_customer.shape}')
print(f'Unique loans: {loan_customer["masked_loan_id"].nunique():,}')
display(loan_customer.head(2))

Overlapping columns (will use suffixes): {'fileno'}



Loan + Customer merged shape: (630632, 30)
Unique loans: 630,632


,fileno,loan_origination_date,scheduled_maturity_date,tenure_total,product_type,secured_flag,interest_rate,emi_frequency,disbursed_amount,principal_outstanding,...,primary_customer_age,primary_gender,primary_employment_status,primary_customer_income,primary_income_to_emi_ratio,customer_segment,prior_loans_count,prior_delinquency_history,customer_geography_type,num_co_applicants
0,5028651,2006-07-03,2009-06-03,36,FSBL LAP,Yes,26.443,Monthly,250000.0,12950.0,...,NaN,NaN,NaN,NaN,NaN,Low,1.0,90+,NaN,0.0
1,5028652,2007-08-24,2010-01-27,30,FSBL LAP,Yes,29.244,Monthly,500000.0,1910.0,...,NaN,NaN,NaN,NaN,NaN,Low,1.0,90+,NaN,0.0


### Filter: Remove loans with no payment records
Loans in the loan table that have zero payment records are excluded — they cannot contribute to payment-level credit risk analysis.

In [12]:
# Remove loans from loan_customer that have NO payment records at all
payment_loan_ids = set(payment_df['masked_loan_id'].unique())
before_count = len(loan_customer)
before_loans = loan_customer['masked_loan_id'].nunique()

loan_customer = loan_customer[loan_customer['masked_loan_id'].isin(payment_loan_ids)]

after_count = len(loan_customer)
after_loans = loan_customer['masked_loan_id'].nunique()

removed_loans = before_loans - after_loans
removed_rows = before_count - after_count

print('=== Filter: Loans without payment records ===')
print(f'Before : {before_count:,} rows | {before_loans:,} loans')
print(f'After  : {after_count:,} rows | {after_loans:,} loans')
print(f'Removed: {removed_rows:,} rows | {removed_loans:,} loans (no payment data)')


=== Filter: Loans without payment records ===
Before : 630,632 rows | 630,632 loans
After  : 601,326 rows | 601,326 loans
Removed: 29,306 rows | 29,306 loans (no payment data)


## 4. Step 3 — Merge with Payment Details

In [13]:
# Clean payment_df: drop unnamed index column
payment_clean = payment_df.drop(columns=[c for c in payment_df.columns if c.startswith('Unnamed')], errors='ignore')

print(f'Payment details shape : {payment_clean.shape}')
print(f'Columns: {payment_clean.columns.tolist()}')

Payment details shape : (10218442, 21)
Columns: ['fileno', 'status', 'closure date', 'maturity date', 'installment number', 'installment_due_date', 'emi_amount', 'payment_date', 'paid EMI', 'unpaid EMI', 'payment_mode', 'bounce_flag', 'bounced date', 'number_of_paid_emis', 'number_of_unpaid_emis', 'dpd (max)', 'bucket_prior', 'bucket_current', 'year', 'month', 'masked_loan_id']


In [14]:
# INNER JOIN: Payment with Loan+Customer
# Only keep payment records that have matching loan+customer data
# This removes orphan payment rows (loans in payment but not in loan table)

common_cols_2 = set(payment_clean.columns) & set(loan_customer.columns) - {'masked_loan_id'}
print(f'Overlapping columns with payment: {common_cols_2}')

before_pymt_rows = len(payment_clean)

master = payment_clean.merge(
    loan_customer,
    on='masked_loan_id',
    how='inner',
    suffixes=('_pymt', '_loancust')
)

# For overlapping columns, keep the payment version and drop loan+customer version
cols_to_drop_2 = [c + '_loancust' for c in common_cols_2 if c + '_loancust' in master.columns]
cols_to_rename_2 = {c + '_pymt': c for c in common_cols_2 if c + '_pymt' in master.columns}

master = master.drop(columns=cols_to_drop_2, errors='ignore')
master = master.rename(columns=cols_to_rename_2)

removed_pymt_rows = before_pymt_rows - len(master)
print(f'\nPayment rows before INNER JOIN : {before_pymt_rows:,}')
print(f'Payment rows removed (no loan) : {removed_pymt_rows:,}')
print(f'Master dataset shape           : {master.shape}')
print(f'Unique loans in master         : {master["masked_loan_id"].nunique():,}')
print(f'\nAll columns ({len(master.columns)}):')
print(master.columns.tolist())

Overlapping columns with payment: {'fileno'}



Payment rows before INNER JOIN : 10,218,442
Payment rows removed (no loan) : 20
Master dataset shape           : (10218422, 49)


Unique loans in master         : 601,326

All columns (49):
['fileno', 'status', 'closure date', 'maturity date', 'installment number', 'installment_due_date', 'emi_amount', 'payment_date', 'paid EMI', 'unpaid EMI', 'payment_mode', 'bounce_flag', 'bounced date', 'number_of_paid_emis', 'number_of_unpaid_emis', 'dpd (max)', 'bucket_prior', 'bucket_current', 'year', 'month', 'masked_loan_id', 'loan_origination_date', 'scheduled_maturity_date', 'tenure_total', 'product_type', 'secured_flag', 'interest_rate', 'emi_frequency', 'disbursed_amount', 'principal_outstanding', 'ltv', 'bureau_score_origination', 'restructured_flag', 'moratorium_flag', 'branch', 'state', 'repayment_start_date', 'Sourcing channel', 'scheme', 'primary_customer_age', 'primary_gender', 'primary_employment_status', 'primary_customer_income', 'primary_income_to_emi_ratio', 'customer_segment', 'prior_loans_count', 'prior_delinquency_history', 'customer_geography_type', 'num_co_applicants']


## 5. QC Checks

In [15]:
print('=' * 60)
print('QC CHECK 1: Row count validation')
print('=' * 60)
print(f'Payment source rows     : {len(payment_clean):,}')
print(f'Master rows (after filter): {len(master):,}')
print(f'Rows removed            : {len(payment_clean) - len(master):,}')
print(f'  - Loans removed from loan_customer (no payment data) : {removed_loans:,}')
print(f'  - Payment rows removed (no loan data, INNER JOIN)    : {removed_pymt_rows:,}')

QC CHECK 1: Row count validation
Payment source rows     : 10,218,442
Master rows (after filter): 10,218,422
Rows removed            : 20
  - Loans removed from loan_customer (no payment data) : 29,306
  - Payment rows removed (no loan data, INNER JOIN)    : 20


In [16]:
print('=' * 60)
print('QC CHECK 2: No duplicate columns')
print('=' * 60)
dup_cols = master.columns[master.columns.duplicated()].tolist()
if dup_cols:
    print(f'  FAIL: Duplicate columns found: {dup_cols}')
else:
    print('  PASS: No duplicate columns')

# Check for leftover suffix columns
suffix_cols = [c for c in master.columns if c.endswith('_pymt') or c.endswith('_loancust') or c.endswith('_loan') or c.endswith('_cust')]
if suffix_cols:
    print(f'  NOTE: Columns with merge suffixes still present: {suffix_cols}')
else:
    print('  PASS: No leftover suffix columns')

QC CHECK 2: No duplicate columns
  PASS: No duplicate columns
  PASS: No leftover suffix columns


In [17]:
print('=' * 60)
print('QC CHECK 3: Spot-check 3 random loans')
print('=' * 60)

# Pick 3 loans that exist in all 3 source tables
common_loans = (
    set(customer_df['masked_loan_id'].unique()) &
    set(loan_df['masked_loan_id'].unique()) &
    set(payment_df['masked_loan_id'].unique())
)
sample_loans = list(common_loans)[:3]
print(f'Checking loans: {sample_loans}\n')

for loan_id in sample_loans:
    print(f'--- Loan: {loan_id} ---')
    
    # Source data
    src_loan = loan_df[loan_df['masked_loan_id'] == loan_id]
    src_pymt = payment_df[payment_df['masked_loan_id'] == loan_id]
    src_cust = customer_df[customer_df['masked_loan_id'] == loan_id]
    master_rows = master[master['masked_loan_id'] == loan_id]
    
    print(f'  Source payment rows: {len(src_pymt)}, Master rows: {len(master_rows)} -> Match: {len(src_pymt) == len(master_rows)}')
    
    # Verify loan fields match
    if len(src_loan) > 0 and len(master_rows) > 0:
        loan_amt = src_loan['disbursed_amount'].values[0]
        master_amt = master_rows['disbursed_amount'].values[0]
        print(f'  Disbursed amount  : source={loan_amt}, master={master_amt} -> Match: {loan_amt == master_amt or (pd.isna(loan_amt) and pd.isna(master_amt))}')
    
    # Verify payment fields match
    if len(src_pymt) > 0 and len(master_rows) > 0:
        pymt_emi = src_pymt['emi_amount'].values[0]
        master_emi = master_rows['emi_amount'].values[0]
        print(f'  EMI amount        : source={pymt_emi}, master={master_emi} -> Match: {pymt_emi == master_emi or (pd.isna(pymt_emi) and pd.isna(master_emi))}')
    print()

QC CHECK 3: Spot-check 3 random loans


Checking loans: ['LN_ARS_JDnt_MU=', 'LN_qDbJPLubw7c=', 'LN_HwPkWOhKLFs=']

--- Loan: LN_ARS_JDnt_MU= ---


  Source payment rows: 22, Master rows: 22 -> Match: True
  Disbursed amount  : source=350000.0, master=350000.0 -> Match: True
  EMI amount        : source=8582.0, master=8582.0 -> Match: True

--- Loan: LN_qDbJPLubw7c= ---


  Source payment rows: 24, Master rows: 24 -> Match: True
  Disbursed amount  : source=300000.0, master=300000.0 -> Match: True
  EMI amount        : source=7547.0, master=7547.0 -> Match: True

--- Loan: LN_HwPkWOhKLFs= ---


  Source payment rows: 7, Master rows: 7 -> Match: True
  Disbursed amount  : source=659426.81, master=659426.81 -> Match: True
  EMI amount        : source=15733.0, master=15733.0 -> Match: True



In [18]:
print('=' * 60)
print('QC CHECK 4: Key coverage')
print('=' * 60)

payment_loans = set(payment_clean['masked_loan_id'].unique())
customer_loans = set(customer_df['masked_loan_id'].unique())
loan_loans = set(loan_clean['masked_loan_id'].unique())

pymt_with_cust = len(payment_loans & customer_loans)
pymt_with_loan = len(payment_loans & loan_loans)

print(f'Payment loans with customer data : {pymt_with_cust:,} / {len(payment_loans):,} ({pymt_with_cust/len(payment_loans)*100:.1f}%)')
print(f'Payment loans with loan data     : {pymt_with_loan:,} / {len(payment_loans):,} ({pymt_with_loan/len(payment_loans)*100:.1f}%)')
print(f'Payment loans missing from loan  : {len(payment_loans - loan_loans):,}')
print(f'Loan loans missing from payment  : {len(loan_loans - payment_loans):,}')

QC CHECK 4: Key coverage


Payment loans with customer data : 601,328 / 601,328 (100.0%)
Payment loans with loan data     : 601,326 / 601,328 (100.0%)
Payment loans missing from loan  : 2
Loan loans missing from payment  : 29,306


In [19]:
print('=' * 60)
print('QC CHECK 5: Column count validation')
print('=' * 60)

# Unique columns from each source (minus join keys and unnamed index cols)
cust_cols = set(customer_prepared.columns)
loan_cols = set(loan_clean.columns)
pymt_cols = set(payment_clean.columns)

# Overlapping keys removed during merge
overlap_1 = (loan_cols & cust_cols) - {'masked_loan_id'}
overlap_2 = (pymt_cols & (loan_cols | cust_cols)) - {'masked_loan_id'}

expected_approx = len(pymt_cols) + len(loan_cols) + len(cust_cols) - len(overlap_1) - len(overlap_2) - 2  # minus 2 for duplicate masked_loan_id
print(f'Payment cols          : {len(pymt_cols)}')
print(f'Loan cols             : {len(loan_cols)}')
print(f'Customer prepared cols: {len(cust_cols)}')
print(f'Overlap loan-cust     : {len(overlap_1)} ({overlap_1})')
print(f'Overlap pymt-rest     : {len(overlap_2)} ({overlap_2})')
print(f'Expected approx cols  : {expected_approx}')
print(f'Actual master cols    : {len(master.columns)}')

QC CHECK 5: Column count validation
Payment cols          : 21
Loan cols             : 20
Customer prepared cols: 12
Overlap loan-cust     : 1 ({'fileno'})
Overlap pymt-rest     : 1 ({'fileno'})
Expected approx cols  : 49
Actual master cols    : 49


In [20]:
print('=' * 60)
print('Null Summary')
print('=' * 60)
null_summary = master.isnull().sum()
null_pct = (master.isnull().sum() / len(master) * 100).round(2)
null_report = pd.DataFrame({'null_count': null_summary, 'null_pct': null_pct})
null_report = null_report[null_report['null_count'] > 0].sort_values('null_pct', ascending=False)
print(f'Columns with nulls: {len(null_report)} / {len(master.columns)}')
display(null_report)

Null Summary


Columns with nulls: 17 / 49


,null_count,null_pct
customer_geography_type,10218422,100.00
bounced date,9999587,97.86
moratorium_flag,9113990,89.19
closure date,8767158,85.80
bureau_score_origination,7364490,72.07
primary_employment_status,1382839,13.53
payment_mode,276732,2.71
payment_date,276732,2.71
primary_customer_income,214711,2.10
bucket_prior,76949,0.75


## 6. Save Master File and Summary Report

In [21]:
# Save master CSV
master_path = os.path.join(OUTPUT_DIR, 'master_payment_level.csv')
master.to_csv(master_path, index=False)
file_size_mb = os.path.getsize(master_path) / (1024 * 1024)
print(f'Master file saved: {master_path}')
print(f'File size: {file_size_mb:.1f} MB')
print(f'Shape: {master.shape}')

Master file saved: /Users/shiva/Downloads/Five_star_customer_data/master_data/master_payment_level.csv
File size: 3154.9 MB
Shape: (10218422, 49)


In [22]:
# Create summary report in Excel
summary_path = os.path.join(OUTPUT_DIR, 'master_data_summary.xlsx')

with pd.ExcelWriter(summary_path, engine='openpyxl') as writer:
    # Sheet 1: Overview
    overview = pd.DataFrame({
        'Metric': [
            'Total Rows',
            'Total Columns',
            'Unique Loans',
            'Source: Customer Rows',
            'Source: Loan Rows',
            'Source: Payment Rows',
            'Row Count Match (vs Payment)',
            'Payment Loans with Customer Data (%)',
            'Payment Loans with Loan Data (%)',
            'File Size (MB)'
        ],
        'Value': [
            len(master),
            len(master.columns),
            master['masked_loan_id'].nunique(),
            len(customer_df),
            len(loan_df),
            len(payment_df),
            'Yes' if len(master) == len(payment_clean) else 'No',
            f'{pymt_with_cust/len(payment_loans)*100:.1f}%',
            f'{pymt_with_loan/len(payment_loans)*100:.1f}%',
            f'{file_size_mb:.1f}'
        ]
    })
    overview.to_excel(writer, sheet_name='Overview', index=False)
    
    # Sheet 2: Column Details
    col_details = pd.DataFrame({
        'Column': master.columns,
        'Dtype': master.dtypes.values,
        'Non_Null_Count': master.notnull().sum().values,
        'Null_Count': master.isnull().sum().values,
        'Null_Pct': (master.isnull().sum() / len(master) * 100).round(2).values,
        'Unique_Values': [master[c].nunique() for c in master.columns],
    })
    col_details.to_excel(writer, sheet_name='Column Details', index=False)
    
    # Sheet 3: Sample Data
    master.head(100).to_excel(writer, sheet_name='Sample Data (100 rows)', index=False)
    
    # Sheet 4: Null Summary
    null_report.to_excel(writer, sheet_name='Null Summary')

print(f'Summary report saved: {summary_path}')

Summary report saved: /Users/shiva/Downloads/Five_star_customer_data/master_data/master_data_summary.xlsx


In [23]:
print('\n' + '=' * 60)
print('MASTER DATA CREATION COMPLETE')
print('=' * 60)
print(f'Master file  : {master_path}')
print(f'Summary report: {summary_path}')
print(f'Shape         : {master.shape}')
print(f'File size     : {file_size_mb:.1f} MB')


MASTER DATA CREATION COMPLETE
Master file  : /Users/shiva/Downloads/Five_star_customer_data/master_data/master_payment_level.csv
Summary report: /Users/shiva/Downloads/Five_star_customer_data/master_data/master_data_summary.xlsx
Shape         : (10218422, 49)
File size     : 3154.9 MB
